# Notes


Selenium needs to be installed:  
```
pip install requests beautifulsoup4
pip install selenium
```




# Selenium Approach

# Scraper

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime
import time
import re
import json
import os


# ---------- CONFIGURATION ----------
MAX_DEPTH = 5
MAX_TWEETS = 30
WAIT_TIME = 1
USERS_PER_PART = 30
MAX_MENTIONS_PER_ACCOUNT = 7

# ---------- GLOBALS ----------
stop_requested = False  # Set this True to stop crawling gracefully (e.g., from another thread or manually in notebook)

# ---------- UTILITY FUNCTIONS ----------
def extract_mentions(text):
    return re.findall(r"@\w+", text)

def save_partial(edges, start_user, max_depth, max_tweets, start_datetime):
    directory = "mention_networks/"
    os.makedirs(directory, exist_ok=True)
    json_name = f"{start_user}_{max_depth}_{max_tweets}-{start_datetime.strftime("%d_%m_%Y-%H-%M-%S")}.json"
    json_path = os.path.join(directory, json_name)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(edges, f, indent=2)
    print(f"\n💾 Saved partial mention graph to {json_path}")

# ---------- TWITTER SCRAPER ----------
def login_to_twitter():
    chrome_options = Options()
    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(options=chrome_options)
    driver.get("https://twitter.com/login")
    print("\n🔐 Please log in manually in the opened browser window.")
    input("✅ Press ENTER after you have successfully logged in...\n")
    return driver

def wait_for_tweets(driver, timeout=10):
    try:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.XPATH, "//article"))
        )
    except:
        print("⚠️ Timeout waiting for tweets to load.")

def scrape_mentions(driver, username):
    url = f"https://twitter.com/{username}"
    print(f"Opening: {url}")
    driver.get(url)
    wait_for_tweets(driver)

    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(WAIT_TIME)

    tweet_texts = {}
    scroll_attempts = 0

    def collect_tweets():
        articles = driver.find_elements(By.XPATH, "//article")
        for article in articles:
            try:
                spans = article.find_elements(By.XPATH, ".//span")
                text = " ".join(span.text for span in spans if span.text.strip())
                if text and text not in tweet_texts:
                    tweet_texts[text] = text
            except:
                continue

    # Initial scroll and collection
    collect_tweets()

    if len(tweet_texts) == 0:
        print("⚠️ No tweets found. Trying Replies tab temporarily...")
        try:
            # Click "Replies" tab
            replies_tab = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.LINK_TEXT, "Replies"))
            )
            replies_tab.click()
            time.sleep(WAIT_TIME)
        except Exception as e:
            print(f"⚠️ Failed to switch tabs: {e}")

    # Continue scrolling and collecting
    while len(tweet_texts) < MAX_TWEETS and scroll_attempts < 4:
        collect_tweets()
        scroll_attempts += 1
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(WAIT_TIME)

    tweets_texts = list(tweet_texts.values())[:MAX_TWEETS]
    print(f"🧪 Found {len(tweets_texts)} unique tweet articles on @{username}")

    mentions = []

    for text in tweets_texts:
        found_mentions = extract_mentions(text)
        for m in found_mentions:
            if m not in mentions:
                mentions.append(m)

    return mentions[:MAX_MENTIONS_PER_ACCOUNT]


# ---------- RECURSIVE CRAWLER ----------
def crawl_mentions_network(start_user, max_depth=MAX_DEPTH, max_tweets=MAX_TWEETS):
    global stop_requested

    driver = login_to_twitter()
    visited = set()
    edges = []
    start_datetime = datetime.now()

    def dfs(user, depth):
        nonlocal start_datetime
        if stop_requested:
            print("⚠️ Stop requested, aborting DFS.")
            return
        if depth > max_depth or user in visited:
            return

        visited.add(user)

        # Periodic saving
        if len(visited) % USERS_PER_PART == 0:
            print(f"\n⏳ Reached {len(visited)} visited users, saving partial results...")
            save_partial(edges, start_user, max_depth, max_tweets, start_datetime)

        try:
            mentions = scrape_mentions(driver, user)
        except Exception as e:
            print(f"❌ Failed to scrape @{user}: {e}")
            return

        for mentioned in mentions:
            if stop_requested:
                print("⚠️ Stop requested, breaking mention loop.")
                return
            mentioned_user = mentioned[1:]  # Remove '@'
            edges.append((user, mentioned_user))
            dfs(mentioned_user, depth + 1)

    try:
        dfs(start_user, 0)
    except KeyboardInterrupt:
        print("\n⚠️ KeyboardInterrupt detected! Stopping safely...")
        stop_requested = True
    except Exception as e:
        print(f"❌ Unexpected exception: {e}")

    print("\n⏳ Saving final results...")
    save_partial(edges, start_user, max_depth, max_tweets, start_datetime)
    driver.quit()
    return edges

# ---------- MAIN ----------
if __name__ == "__main__":
    start_username = "QuantumFracture"  # Replace with your starting username

    try:
        mention_edges = crawl_mentions_network(start_username)
    except KeyboardInterrupt:
        print("\n⚠️ KeyboardInterrupt detected in main! Exiting safely...")

    print("\n📈 Mention Edges:")
    for source, target in mention_edges:
        print(f"{source} -> {target}")

# Test



🔐 Please log in manually in the opened browser window.
Opening: https://twitter.com/QuantumFracture
🧪 Found 25 unique tweet articles on @QuantumFracture
Opening: https://twitter.com/ICFOnians
🧪 Found 21 unique tweet articles on @ICFOnians
Opening: https://twitter.com/LuxQuanta
🧪 Found 24 unique tweet articles on @LuxQuanta
Opening: https://twitter.com/EstudiantesRSEF
🧪 Found 22 unique tweet articles on @EstudiantesRSEF
Opening: https://twitter.com/s
🧪 Found 30 unique tweet articles on @s
Opening: https://twitter.com/KSI
🧪 Found 25 unique tweet articles on @KSI
Opening: https://twitter.com/estuRSEF_UGR
🧪 Found 21 unique tweet articles on @estuRSEF_UGR
Opening: https://twitter.com/fcienciasugr
🧪 Found 23 unique tweet articles on @fcienciasugr
Opening: https://twitter.com/FundacionAreces
🧪 Found 21 unique tweet articles on @FundacionAreces
Opening: https://twitter.com/FisicasUGR
🧪 Found 27 unique tweet articles on @FisicasUGR
Opening: https://twitter.com/javiroji_03
🧪 Found 24 unique twe

# Number of Nodes

In [5]:
import json

with open('mention_networks/network_mega_merge.json', 'r', encoding='utf-8') as f:
    edges = json.load(f)

unique_nodes = set()    # Sets only store unique values
for source, target in edges:
    unique_nodes.add(source)
    unique_nodes.add(target)

print(f"Number of unique nodes: {len(unique_nodes)}")

Number of unique nodes: 8997


# Interactive Graph

In [6]:
import json
import networkx as nx
import plotly.graph_objects as go

# Load your edges from JSON or define them here
with open("mention_networks/eldonutjaja_6_30-30_05_2025-08-42-37.json", "r", encoding="utf-8") as f:
    edges = json.load(f)

# Build directed graph
G = nx.DiGraph()
G.add_edges_from(edges)

start_user = "eldonutjaja"

# Compute shortest path length from start_user
depths = dict(nx.single_source_shortest_path_length(G, start_user))

# Compute positions
pos = nx.spring_layout(G, k=0.5, seed=42)

# Extract node positions
x_nodes = [pos[node][0] for node in G.nodes()]
y_nodes = [pos[node][1] for node in G.nodes()]

# Edge traces
edge_x = []
edge_y = []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines')

# Node sizes scaled by depth, with fixed size for start_user
node_sizes = []
for node in G.nodes():
    if node == start_user:
        node_sizes.append(200)  # Larger size for start node
    else:
        depth = depths.get(node, max(depths.values()) + 1)
        size = 100 / (depth + 1)  # Scale size by inverse of depth+1
        node_sizes.append(size)

# Node trace
node_trace = go.Scatter(
    x=x_nodes, y=y_nodes,
    mode='markers+text',
    text=[str(node) for node in G.nodes()],
    textposition="top center",
    hoverinfo='text',
    marker=dict(
        showscale=False,
        color=["lightcoral" if node == start_user else "skyblue" for node in G.nodes()],
        size=node_sizes,
        line_width=2))

fig = go.Figure(data=[edge_trace, node_trace],
                layout=go.Layout(
                    title="Interactive Twitter Mention Network",
                    title_font_size=16,
                    showlegend=False,
                    hovermode='closest',
                    margin=dict(b=20, l=5, r=5, t=40),
                    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                )
fig.show()


ModuleNotFoundError: No module named 'plotly'

In [7]:
def find_and_visualize_path(G, source_user, target_user):
    try:
        # Find the shortest path
        path = nx.shortest_path(G, source=source_user, target=target_user)
        print(f"✅ Path found from @{source_user} to @{target_user}:")
        print(" → ".join(path))
    except nx.NetworkXNoPath:
        print(f"❌ No path found from @{source_user} to @{target_user}.")
        return
    except nx.NodeNotFound as e:
        print(f"❌ Node not found: {e}")
        return

    # Create subgraph from path
    path_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
    path_graph = nx.DiGraph()
    path_graph.add_edges_from(path_edges)

    # Layout for consistent node positions
    pos = nx.spring_layout(path_graph, k=0.5, seed=42)

    x_nodes = [pos[node][0] for node in path_graph.nodes()]
    y_nodes = [pos[node][1] for node in path_graph.nodes()]

    edge_x = []
    edge_y = []
    for edge in path_graph.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=2, color='black'),
        hoverinfo='none',
        mode='lines')

    # Color and size nodes
    node_trace = go.Scatter(
        x=x_nodes, y=y_nodes,
        mode='markers+text',
        text=[str(node) for node in path_graph.nodes()],
        textposition="top center",
        hoverinfo='text',
        marker=dict(
            showscale=False,
            color=["lightcoral" if node == source_user else "orange" if node == target_user else "lightgreen" for node in path_graph.nodes()],
            size=[25 for _ in path_graph.nodes()],
            line_width=2))

    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(
                        title=f"Path from @{source_user} to @{target_user}",
                        title_font_size=16,
                        showlegend=False,
                        hovermode='closest',
                        margin=dict(b=20, l=5, r=5, t=40),
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                    )
    fig.show()


In [8]:
find_and_visualize_path(G, "WgPaulo", "elonmusk")

NameError: name 'G' is not defined

In [7]:
def visualize_user_network(G, user, max_depth=3):
    if user not in G:
        print(f"❌ User @{user} not found in the network.")
        return

    # Use BFS to find all reachable nodes up to max_depth
    bfs_edges = list(nx.bfs_edges(G, user, depth_limit=max_depth))
    sub_nodes = {user}
    for u, v in bfs_edges:
        sub_nodes.add(u)
        sub_nodes.add(v)

    # Create the subgraph
    subgraph = G.subgraph(sub_nodes).copy()

    print(f"📌 Showing local mention network for @{user} (depth: {max_depth}) with {len(subgraph.nodes)} nodes.")

    # Layout positions
    pos = nx.spring_layout(subgraph, k=0.5, seed=42)
    x_nodes = [pos[node][0] for node in subgraph.nodes()]
    y_nodes = [pos[node][1] for node in subgraph.nodes()]

    # Edge traces
    edge_x = []
    edge_y = []
    for edge in subgraph.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=1.5, color='gray'),
        hoverinfo='none',
        mode='lines'
    )

    # Node trace
    node_trace = go.Scatter(
        x=x_nodes, y=y_nodes,
        mode='markers+text',
        text=[str(node) for node in subgraph.nodes()],
        textposition="top center",
        hoverinfo='text',
        marker=dict(
            showscale=False,
            color=["lightcoral" if node == user else "skyblue" for node in subgraph.nodes()],
            size=[25 if node == user else 15 for node in subgraph.nodes()],
            line_width=2
        )
    )

    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(
                        title=f"Local Network of @{user} (up to depth {max_depth})",
                        title_font_size=16,
                        showlegend=False,
                        hovermode='closest',
                        margin=dict(b=20, l=5, r=5, t=40),
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                    )
    fig.show()


In [25]:
visualize_user_network(G, "DrKuchoGames", max_depth=1)

📌 Showing local mention network for @DrKuchoGames (depth: 1) with 1 nodes.
